In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, col, round, desc

print("=" * 50)
print("DAY 2 — IPL Pipeline with PySpark")
print("=" * 50)

DAY 2 — IPL Pipeline with PySpark


In [2]:
# Start Spark
spark = SparkSession.builder \
    .appName("IPL-Pipeline") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")  # hides noisy logs

In [3]:
# ── EXTRACT ──────────────────────────────────────────
print("\n[EXTRACT] Reading CSV...")
df = spark.read.csv(
    "/home/jovyan/matches.csv",
    header=True,
    inferSchema=True
)
print(f" Rows loaded: {df.count()}")
df.printSchema()


[EXTRACT] Reading CSV...
 Rows loaded: 1095
root
 |-- id: integer (nullable = true)
 |-- season: string (nullable = true)
 |-- city: string (nullable = true)
 |-- date: date (nullable = true)
 |-- match_type: string (nullable = true)
 |-- player_of_match: string (nullable = true)
 |-- venue: string (nullable = true)
 |-- team1: string (nullable = true)
 |-- team2: string (nullable = true)
 |-- toss_winner: string (nullable = true)
 |-- toss_decision: string (nullable = true)
 |-- winner: string (nullable = true)
 |-- result: string (nullable = true)
 |-- result_margin: string (nullable = true)
 |-- target_runs: string (nullable = true)
 |-- target_overs: string (nullable = true)
 |-- super_over: string (nullable = true)
 |-- method: string (nullable = true)
 |-- umpire1: string (nullable = true)
 |-- umpire2: string (nullable = true)



In [4]:
# ── TRANSFORM ────────────────────────────────────────
print("\n[TRANSFORM] Running analysis...")

# 1. Most match wins per team
print("\n--- Top 10 Teams by Wins ---")
wins = df.filter(col("winner").isNotNull()) \
         .groupBy("winner") \
         .agg(count("*").alias("total_wins")) \
         .orderBy(desc("total_wins"))
wins.show(10)


[TRANSFORM] Running analysis...

--- Top 10 Teams by Wins ---
+--------------------+----------+
|              winner|total_wins|
+--------------------+----------+
|      Mumbai Indians|       144|
| Chennai Super Kings|       138|
|Kolkata Knight Ri...|       131|
|Royal Challengers...|       116|
|    Rajasthan Royals|       112|
| Sunrisers Hyderabad|        88|
|     Kings XI Punjab|        88|
|    Delhi Daredevils|        67|
|      Delhi Capitals|        48|
|     Deccan Chargers|        29|
+--------------------+----------+
only showing top 10 rows



In [11]:
# 2. Toss winner vs match winner — does winning toss help?
import builtins


print("\n--- Toss Winner = Match Winner? ---")
toss_match = df.filter(col("winner").isNotNull()) \
               .withColumn(
                   "toss_helped",
                   (col("toss_winner") == col("winner")).cast("integer")
               )
total = toss_match.count()
helped = toss_match.filter(col("toss_helped") == 1).count()
pct = int(builtins.round((helped / total) * 100, 2))
print(f" Toss winner won the match: {helped}/{total} times ({pct}%)")


--- Toss Winner = Match Winner? ---
 Toss winner won the match: 554/1095 times (50%)


In [12]:
# 3. Matches per season
print("\n--- Matches Per Season ---")
df.groupBy("season") \
  .agg(count("*").alias("matches_played")) \
  .orderBy("season") \
  .show(20)


--- Matches Per Season ---
+-------+--------------+
| season|matches_played|
+-------+--------------+
|2007/08|            58|
|   2009|            57|
|2009/10|            60|
|   2011|            73|
|   2012|            74|
|   2013|            76|
|   2014|            60|
|   2015|            59|
|   2016|            60|
|   2017|            59|
|   2018|            60|
|   2019|            60|
|2020/21|            60|
|   2021|            60|
|   2022|            74|
|   2023|            74|
|   2024|            71|
+-------+--------------+



In [13]:
# ── LOAD ─────────────────────────────────────────────
print("\n[LOAD] Writing output to Parquet...")
wins.write.mode("overwrite").parquet("output/team_wins")
print(" Written to output/team_wins/")


[LOAD] Writing output to Parquet...
 Written to output/team_wins/


In [14]:
spark.stop()
print("\nDay 2 complete. PySpark pipeline ran successfully.")


Day 2 complete. PySpark pipeline ran successfully.
